## Packages

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
np.random.seed(1)

## Data

In [2]:
# Load CSV files
# Train Data
trainop = pd.read_csv('../data/2024-34-1/data/train_operational_readouts.csv')
trainspec = pd.read_csv('../data/2024-34-1/data/train_specifications.csv')
traintte = pd.read_csv('../data/2024-34-1/data/train_tte.csv')
# Validation Data
vallab = pd.read_csv('../data/2024-34-1/data/validation_labels.csv')
valop = pd.read_csv('../data/2024-34-1/data/validation_operational_readouts.csv')
valspec = pd.read_csv('../data/2024-34-1/data/validation_specifications.csv')
# Test Data
testlab = pd.read_csv('../data/2024-34-1/data/test_labels.csv')
testop = pd.read_csv('../data/2024-34-1/data/test_operational_readouts.csv')
testspec = pd.read_csv('../data/2024-34-1/data/test_specifications.csv')

# Convert to DataFrames
# Train Data
df_trainop = pd.DataFrame(data=trainop)
df_trainspec = pd.DataFrame(data=trainspec)
df_traintte = pd.DataFrame(data=traintte)
# Validation Data
df_vallab = pd.DataFrame(data=vallab)
df_valop = pd.DataFrame(data=valop)
df_valspec = pd.DataFrame(data=valspec)
# Test Data
df_testlab = pd.DataFrame(data=testlab)
df_testop = pd.DataFrame(data=testop)
df_testspec = pd.DataFrame(data=testspec)

# Index
trainop_idx = df_trainop.set_index("vehicle_id").sort_index()
trainop_idx['time_step'] = trainop_idx['time_step'].astype('float32')
testop_idx = df_testop.set_index("vehicle_id").sort_index()
testop_idx['time_step'] = testop_idx['time_step'].astype('float32')
valop_idx = df_valop.set_index("vehicle_id").sort_index()
valop_idx['time_step'] = valop_idx['time_step'].astype('float32')

In [3]:
last_step = df_trainop.groupby("vehicle_id")["time_step"].max()

repaired = df_traintte[df_traintte["in_study_repair"] == 1].copy()
repaired["last_step"] = repaired["vehicle_id"].map(last_step)
repaired["gap"] = repaired["length_of_study_time_step"] - repaired["last_step"]
repaired["gap"].describe()

count    2272.000000
mean        3.873151
std        11.763505
min         0.200000
25%         0.600000
50%         1.800000
75%         4.000000
max       260.000000
Name: gap, dtype: float64

#### Findings:
- All in study time steps are greater than last time given in the train operational readouts data

## Label Generation

![alt text](images/image(5).png)

Equipment health formula: $f(x) = t{\text{failure}_v} - t{\text{step}_v}_i$

where:

$f(x) > 48 =$ class 0

$f(x) <= 48 =$ class 1

$f(x) <= 24 =$ class 2

$f(x) <= 12 =$ class 3

$f(x) <= 6 =$ class 4


In [4]:
def EHI(tfailure, tstepvi):
    """ Computes the time at failue
        at vehicle v minus the time step at vehicle v's ith observation
        and assigns the appropriate class for equipment health.

        Args:
            tfailure (Float): The time the vehicle was last recorded if repaired.
            tsepvi (Float): The current time step of the vehicle.

        Returns:
            ehiclass (Integer): A number spanning 0 to 4 representing a class.
    """

    fx = tfailure - tstepvi

    if fx <= 6:
        ehiclass = 4
    elif fx <= 12:
        ehiclass = 3
    elif fx <= 24:
        ehiclass = 2
    elif fx <= 48:
        ehiclass = 1
    else:
        ehiclass = 0

    return ehiclass

In [5]:
df_labeledtrain = df_trainop.copy()
print("df_trainop shape:", df_trainop.shape)
print("df_labeledtrain shape:", df_labeledtrain.shape)

df_trainop shape: (1122452, 107)
df_labeledtrain shape: (1122452, 107)


In [6]:
failure_time = df_traintte.set_index("vehicle_id")["length_of_study_time_step"]
repair_flag = df_traintte.set_index("vehicle_id")["in_study_repair"]

labels = np.array([
    EHI(failure_time[row.vehicle_id], row.time_step)
    if repair_flag[row.vehicle_id] == 1 else 0
    for row in df_trainop.itertuples()
], dtype=np.int8)

In [7]:
print(labels[:172], '\n')  # This is vehicle 0 and it had no repair performed.
print(labels[546:596])  # This is vehicle 22 and it had a repair performed.

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] 

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 1 1 1 1 1 2 2 2 3 3 3 4]


#### As the time readouts progress the equiment health indicator changes overtime if the vehicle was flagged as repaired.

In [8]:
df_labeledtrain['class_label'] = labels
df_labeledtrain.head()

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,167_4,167_5,167_6,167_7,167_8,167_9,309_0,272_0,272_1,272_2,272_3,272_4,272_5,272_6,272_7,272_8,272_9,835_0,370_0,291_0,291_1,291_2,291_3,291_4,291_5,291_6,291_7,291_8,291_9,291_10,158_0,158_1,158_2,158_3,158_4,158_5,158_6,158_7,158_8,158_9,100_0,459_0,459_1,459_2,459_3,459_4,459_5,459_6,459_7,459_8,459_9,459_10,459_11,459_12,459_13,459_14,459_15,459_16,459_17,459_18,459_19,397_0,397_1,397_2,397_3,397_4,397_5,397_6,397_7,397_8,397_9,397_10,397_11,397_12,397_13,397_14,397_15,397_16,397_17,397_18,397_19,397_20,397_21,397_22,397_23,397_24,397_25,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35,class_label
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,1269525.0,4772940.0,2706706.0,222225.0,6240.0,0.0,70.0,1435083.0,857662.0,384579.0,668642.0,7239843.0,398490.0,3887.0,0.0,0.0,0.0,8036751.0,0.0,1227.0,555.0,463.0,925.0,468.0,225.0,535.0,516.0,492.0,729.0,66.0,97056.0,2690052.0,2945268.0,788437.0,687480.0,595164.0,491232.0,532932.0,809628.0,505693.0,858410.0,203.676778,111.911500,147.265389,200.479944,230.306278,277.722417,315.748806,372.164528,864.246250,920.881111,637.901639,744.618944,880.866889,1272.323972,1847.623667,940.785694,2.900083,0.208444,0.056417,0.058444,446956.0,411420.0,203024.0,26636.0,29156.0,7616.0,449537.0,233352.0,139920.0,12648.0,2813.0,224.0,53161.0,178881.0,138250.0,13328.0,3581.0,88.0,16361.0,131601.0,116541.0,13506.0,2856.0,48.0,6337.0,105412.0,95728.0,15609.0,1984.0,8.0,784.0,150228.0,261904.0,93172.0,17874.0,452.0,0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,1269526.0,4772940.0,2706706.0,222225.0,6240.0,0.0,70.0,1440661.0,857662.0,384579.0,668642.0,7239843.0,398490.0,3887.0,0.0,0.0,0.0,8040811.0,0.0,1230.0,558.0,463.0,925.0,469.0,226.0,535.0,516.0,493.0,729.0,66.0,97056.0,2693100.0,2947368.0,788437.0,687480.0,595164.0,491232.0,532932.0,809628.0,505693.0,860571.0,204.256750,112.924250,147.265389,201.479944,230.306278,277.722417,315.748806,372.164528,864.246250,920.881111,637.901639,745.618944,880.866889,1272.323972,1847.623667,940.785694,2.900083,0.208444,0.056417,0.058444,446964.0,411420.0,203027.0,26638.0,29157.0,7616.0,451193.0,233354.0,139920.0,12649.0,2813.0,224.0,53210.0,178883.0,138252.0,13328.0,3582.0,88.0,16368.0,131601.0,116542.0,13507.0,2856.0,48.0,6339.0,105413.0,95729.0,15610.0,1984.0,8.0,784.0,150228.0,261905.0,93172.0,17874.0,452.0,0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,1787736.0,1133132.0,598351.0,1167062.0,12314224.0,460240.0,3887.0,0.0,0.0,0.0,12777022.0,0.0,2136.0,954.0,850.0,1420.0,722.0,412.0,880.0,666.0,586.0,1143.0,162.0,181632.0,4249020.0,4630440.0,1539133.0,1421172.0,1039764.0,749472.0,740724.0,995796.0,574045.0,1379191.0,321.671972,157.312500,193.792833,263.577611,310.711861,366.149250,415.642472,484.391167,1146.111611,1286.536333,900.062917,1123.232556,1449.545611,2140.037472,5046.748278,1151.010139,3.320194,0.218806,0.056778,0.058444,756665.0,647348.0,286811.0,30967.0,31213.0,7745.0,633790.0,423395.0,271940.0,16190.0,3573.0,232.0,75038.0,352791.0,327992.0,17325.0,4451.0,92.0,24028.0,234737.0,216619.0,17000.0,3476.0,48.0,12055.0,167693.0,142900.0,19263.0,2441.0,12.0,1420.0,204832.0,313485.0,106464.0,19306.0,452.0,0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,1824409.0,1166074.0,634595.0,1233908.0,13275730.0,466753.0,3887.0,0.0,0.0,0.0,13612083.0,0.0,2218.0,1014.0,892.0,1471.0,749.0,425.0,901.0,702.0,589.0,1197.0,174.0,193728.0,4462548.0,4988028.0,1696022.0,1565484.0,1112544.0,789228.0,774588.0,1015104.0,576901.0,1428606.0,331.479028,162.731639,198.104472,269.712889,320.087333,377.478667,425.901361,495.749583,1173.882583,1323.460972,923.099361,1161.893139,1501.973944,2208.782833,5587.856667,1160.593833,3.336417,0.218806,0.056778,0.058444,812577.0,686860.0,302955.0,31927.0,31488.0,7749.0,651902.0,478279.0,292109.0,16755.0,3753.0,2

In [9]:
df_trainspec.head()

,vehicle_id,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7
0,0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0
1,2,Cat0,Cat1,Cat1,Cat0,Cat0,Cat0,Cat0,Cat1
2,3,Cat0,Cat1,Cat1,Cat1,Cat0,Cat0,Cat0,Cat1
3,4,Cat0,Cat0,Cat2,Cat1,Cat0,Cat0,Cat0,Cat1
4,5,Cat0,Cat2,Cat2,Cat0,Cat0,Cat0,Cat0,Cat1


In [10]:
df_trainspec = df_trainspec.set_index('vehicle_id')
df_valspec = df_valspec.set_index('vehicle_id')
df_testspec = df_testspec.set_index('vehicle_id')
df_trainspec.head()

,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7
vehicle_id,,,,,,,,
0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0
2,Cat0,Cat1,Cat1,Cat0,Cat0,Cat0,Cat0,Cat1
3,Cat0,Cat1,Cat1,Cat1,Cat0,Cat0,Cat0,Cat1
4,Cat0,Cat0,Cat2,Cat1,Cat0,Cat0,Cat0,Cat1
5,Cat0,Cat2,Cat2,Cat0,Cat0,Cat0,Cat0,Cat1


## Temporal Aggregates
why temporal aggregates:

- Because raw value at one moment doesn't tell the model anything about trajectory and trajectory is what is needed to make predictions in this case.                 
- Sensor read outs trend near linear with degradation = rate of change is the likely one of the most predictive signals.
- XGBoost has no memory across rows so any trajectory information has to be precomputed into the row as an explicit feature.

each sensor gets its own last value, mean, std, slope and delta. class label remains unchanged 

In [11]:
def add_temporal_aggregates(df_indexed, sensor_cols):
    """
    Adds expanding-window temporal aggregate features (last, mean, std,
    slope, delta) for every sensor column, computed per vehicle using
    only that vehicle's readings up to and including each row -- never
    future readings, to avoid leakage into earlier-timestep labels.

    All new columns are built in a dict and attached to the dataframe in
    a single concat at the end, rather than assigned one at a time in the
    loop. Assigning columns individually inside a loop over 100+ sensors
    forces pandas to repeatedly reallocate the whole frame's memory
    layout, which is slow and can trigger a "highly fragmented dataframe"
    warning at this dataset's scale -- a single batched concat avoids it.

    Slope is computed via closed-form cumulative sums (a standard
    least-squares trick) rather than .expanding().apply(), since the
    latter calls a Python function once per row per sensor -- far too
    slow at 1.1M+ rows x 105 sensors.

    Args:
        df_indexed (DataFrame): operational readouts, indexed by
            vehicle_id, MUST already be sorted by vehicle_id then
            time_step (expanding logic assumes row order is chronological
            within each vehicle).
        sensor_cols (list of str): sensor column names to aggregate.

    Returns:
        DataFrame: original columns preserved, plus 5 new columns per
            sensor (e.g. "397_0_last", "397_0_mean", "397_0_std",
            "397_0_slope", "397_0_delta").
    """
    df_indexed[sensor_cols] = df_indexed[sensor_cols].astype("float32")

    grouped_by_vehicle = df_indexed.groupby(level=0)
    new_columns = {}  # collect everything here first, attach once at the end

    for sensor in sensor_cols:
        values = df_indexed[sensor]
        valid = values.notna()

        new_columns[f"{sensor}_last"] = values

        new_columns[f"{sensor}_mean"] = grouped_by_vehicle[sensor].expanding().mean().values
        new_columns[f"{sensor}_std"] = grouped_by_vehicle[sensor].expanding().std().values

        first_value = df_indexed.groupby(level=0)[sensor].transform("first")
        new_columns[f"{sensor}_delta"] = values - first_value

        time_step = df_indexed["time_step"]
        x = time_step.where(valid, 0)
        y = values.where(valid, 0)
        xy = (time_step * values).where(valid, 0)
        xx = (time_step ** 2).where(valid, 0)
        n = valid.astype(int)

        tmp = pd.DataFrame({"n": n, "x": x, "y": y, "xy": xy, "xx": xx}, index=df_indexed.index)
        cum = tmp.groupby(level=0).cumsum()

        numerator = cum["n"] * cum["xy"] - cum["x"] * cum["y"]
        denominator = cum["n"] * cum["xx"] - cum["x"] ** 2
        slope = numerator / denominator
        slope[(cum["n"] < 2) | (denominator == 0)] = np.nan

        new_columns[f"{sensor}_slope"] = slope

    # attach all 525 new columns in a single operation, instead of 525
    # separate ones -- this is what actually avoids the fragmentation warning
    aggregates_df = pd.DataFrame(new_columns, index=df_indexed.index)
    df_indexed = pd.concat([df_indexed, aggregates_df], axis=1)

    return df_indexed

In [12]:
# Temporal aggregates and labels for train set
sensorcols = df_trainop.iloc[:, 2:].columns
sensors = [col for col in sensorcols]
df_trainop_temporal = add_temporal_aggregates(trainop_idx, sensors)
df_trainop_temporal = df_trainop_temporal.join(df_trainspec)
df_trainop_temporal['class_label'] = labels

for col in df_trainspec.columns:
    df_trainop_temporal[col] = df_trainop_temporal[col].astype('category')

print(df_trainop_temporal.memory_usage(deep=True).sum() / 1e9, "GB")

df_trainop_temporal.head()

3.795018783 GB


,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,167_4,167_5,167_6,167_7,167_8,167_9,309_0,272_0,272_1,272_2,272_3,272_4,272_5,272_6,272_7,272_8,272_9,835_0,370_0,291_0,291_1,291_2,291_3,291_4,291_5,291_6,291_7,291_8,291_9,291_10,158_0,158_1,158_2,158_3,158_4,158_5,158_6,158_7,158_8,158_9,100_0,459_0,459_1,459_2,459_3,459_4,459_5,459_6,459_7,459_8,459_9,459_10,459_11,459_12,459_13,459_14,459_15,459_16,459_17,459_18,459_19,397_0,397_1,397_2,397_3,397_4,397_5,397_6,397_7,397_8,397_9,397_10,397_11,397_12,397_13,397_14,397_15,397_16,397_17,397_18,397_19,397_20,397_21,397_22,397_23,397_24,397_25,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35,171_0_last,171_0_mean,171_0_std,171_0_delta,171_0_slope,666_0_last,666_0_mean,666_0_std,666_0_delta,666_0_slope,427_0_last,427_0_mean,427_0_std,427_0_delta,427_0_slope,837_0_last,837_0_mean,837_0_std,837_0_delta,837_0_slope,167_0_last,167_0_mean,167_0_std,167_0_delta,167_0_slope,167_1_last,167_1_mean,167_1_std,167_1_delta,167_1_slope,167_2_last,167_2_mean,167_2_std,167_2_delta,167_2_slope,167_3_last,167_3_mean,167_3_std,167_3_delta,167_3_slope,167_4_last,167_4_mean,167_4_std,167_4_delta,167_4_slope,167_5_last,167_5_mean,167_5_std,167_5_delta,167_5_slope,167_6_last,167_6_mean,167_6_std,167_6_delta,167_6_slope,167_7_last,167_7_mean,167_7_std,167_7_delta,167_7_slope,167_8_last,167_8_mean,167_8_std,167_8_delta,167_8_slope,167_9_last,167_9_mean,167_9_std,167_9_delta,167_9_slope,309_0_last,309_0_mean,309_0_std,309_0_delta,309_0_slope,272_0_last,272_0_mean,272_0_std,272_0_delta,272_0_slope,272_1_last,272_1_mean,272_1_std,272_1_delta,272_1_slope,272_2_last,272_2_mean,272_2_std,272_2_delta,272_2_slope,272_3_last,272_3_mean,272_3_std,272_3_delta,272_3_slope,272_4_last,272_4_mean,272_4_std,272_4_delta,272_4_slope,272_5_last,272_5_mean,272_5_std,272_5_delta,272_5_slope,272_6_last,272_6_mean,272_6_std,272_6_delta,272_6_slope,272_7_last,272_7_mean,272_7_std,272_7_delta,272_7_slope,272_8_last,272_8_mean,272_8_std,272_8_delta,272_8_slope,272_9_last,272_9_mean,272_9_std,272_9_delta,272_9_slope,835_0_last,835_0_mean,835_0_std,835_0_delta,835_0_slope,370_0_last,370_0_mean,370_0_std,370_0_delta,370_0_slope,291_0_last,291_0_mean,291_0_std,291_0_delta,291_0_slope,291_1_last,291_1_mean,291_1_std,291_1_delta,291_1_slope,291_2_last,291_2_mean,291_2_std,291_2_delta,291_2_slope,291_3_last,291_3_mean,291_3_std,291_3_delta,291_3_slope,291_4_last,291_4_mean,291_4_std,291_4_delta,291_4_slope,291_5_last,291_5_mean,291_5_std,291_5_delta,291_5_slope,291_6_last,291_6_mean,291_6_std,291_6_delta,291_6_slope,291_7_last,291_7_mean,291_7_std,291_7_delta,291_7_slope,291_8_last,291_8_mean,291_8_std,291_8_delta,291_8_slope,291_9_last,291_9_mean,291_9_std,291_9_delta,291_9_slope,291_10_last,291_10_mean,291_10_std,291_10_delta,291_10_slope,158_0_last,158_0_mean,158_0_std,158_0_delta,158_0_slope,158_1_last,158_1_mean,158_1_std,158_1_delta,158_1_slope,158_2_last,158_2_mean,158_2_std,158_2_delta,158_2_slope,158_3_last,158_3_mean,158_3_std,158_3_delta,158_3_slope,158_4_last,158_4_mean,158_4_std,158_4_delta,158_4_slope,158_5_last,158_5_mean,158_5_std,158_5_delta,158_5_slope,158_6_last,158_6_mean,158_6_std,158_6_delta,158_6_slope,158_7_last,158_7_mean,158_7_std,158_7_delta,158_7_slope,158_8_last,158_8_mean,158_8_std,158_8_delta,158_8_slope,158_9_last,158_9_mean,158_9_std,158_9_delta,158_9_slope,100_0_last,100_0_mean,100_0_std,100_0_delta,100_0_slope,459_0_last,459_0_mean,459_0_std,459_0_delta,459_0_slope,459_1_last,459_1_mean,459_1_std,459_1_delta,459_1_slope,459_2_last,459_2_mean,459_2_std,459_2_delta,459_2_slope,459_3_last,459_3_mean,459_3_std,459_3_delta,459_3_slope,459_4_last,459_4_mean,459_4_std,459_4_delta,459_4_slope,459_5_last,459_5_mean,459_5_std,459_5_delta,459_5_slope,459_6_last,459_6_mean,459_6_std,459_6_delta,459_6_slope,459_7_last,459_7_mean,459_7_std,459_7_delta,459_7_slope,459_8_last,459_8_mean,459_8_std,459_8_delta,459_8_slope,459_9_last,459_9_mean,459_9_std,459_9_delta,459_9_slope,459_10_last,4

##### Note to self: 
The temporal aggregates in the training set include trend data that the val and test datasets do not, because the Scania Component X challenge excludes it and provides only the equipment health indicator at the time of the final i'th sensor readout. It might be useful to split the temporal training set into an internal validation set to test trend tracking accuracy in the training loop.

In [13]:
# Temporal aggregates and labels for validation set
sensorcols = df_valop.iloc[:, 2:].columns
sensors = [col for col in sensorcols]
df_valop_temporal = add_temporal_aggregates(valop_idx, sensors)
mask = df_valop_temporal.groupby(level=0).cumcount(ascending=False) == 0
df_valop_temporal = df_valop_temporal.join(df_valspec)
df_valop_temporal['class_label'] = np.nan
df_valop_temporal.loc[mask, 'class_label'] = np.asarray(df_vallab['class_label'])

for col in df_valspec.columns:
    df_valop_temporal[col] = df_valop_temporal[col].astype('category')

print(df_valop_temporal.memory_usage(deep=True).sum() / 1e9, "GB")
df_valop_temporal.head()

0.747240863 GB


,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,167_4,167_5,167_6,167_7,167_8,167_9,309_0,272_0,272_1,272_2,272_3,272_4,272_5,272_6,272_7,272_8,272_9,835_0,370_0,291_0,291_1,291_2,291_3,291_4,291_5,291_6,291_7,291_8,291_9,291_10,158_0,158_1,158_2,158_3,158_4,158_5,158_6,158_7,158_8,158_9,100_0,459_0,459_1,459_2,459_3,459_4,459_5,459_6,459_7,459_8,459_9,459_10,459_11,459_12,459_13,459_14,459_15,459_16,459_17,459_18,459_19,397_0,397_1,397_2,397_3,397_4,397_5,397_6,397_7,397_8,397_9,397_10,397_11,397_12,397_13,397_14,397_15,397_16,397_17,397_18,397_19,397_20,397_21,397_22,397_23,397_24,397_25,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35,171_0_last,171_0_mean,171_0_std,171_0_delta,171_0_slope,666_0_last,666_0_mean,666_0_std,666_0_delta,666_0_slope,427_0_last,427_0_mean,427_0_std,427_0_delta,427_0_slope,837_0_last,837_0_mean,837_0_std,837_0_delta,837_0_slope,167_0_last,167_0_mean,167_0_std,167_0_delta,167_0_slope,167_1_last,167_1_mean,167_1_std,167_1_delta,167_1_slope,167_2_last,167_2_mean,167_2_std,167_2_delta,167_2_slope,167_3_last,167_3_mean,167_3_std,167_3_delta,167_3_slope,167_4_last,167_4_mean,167_4_std,167_4_delta,167_4_slope,167_5_last,167_5_mean,167_5_std,167_5_delta,167_5_slope,167_6_last,167_6_mean,167_6_std,167_6_delta,167_6_slope,167_7_last,167_7_mean,167_7_std,167_7_delta,167_7_slope,167_8_last,167_8_mean,167_8_std,167_8_delta,167_8_slope,167_9_last,167_9_mean,167_9_std,167_9_delta,167_9_slope,309_0_last,309_0_mean,309_0_std,309_0_delta,309_0_slope,272_0_last,272_0_mean,272_0_std,272_0_delta,272_0_slope,272_1_last,272_1_mean,272_1_std,272_1_delta,272_1_slope,272_2_last,272_2_mean,272_2_std,272_2_delta,272_2_slope,272_3_last,272_3_mean,272_3_std,272_3_delta,272_3_slope,272_4_last,272_4_mean,272_4_std,272_4_delta,272_4_slope,272_5_last,272_5_mean,272_5_std,272_5_delta,272_5_slope,272_6_last,272_6_mean,272_6_std,272_6_delta,272_6_slope,272_7_last,272_7_mean,272_7_std,272_7_delta,272_7_slope,272_8_last,272_8_mean,272_8_std,272_8_delta,272_8_slope,272_9_last,272_9_mean,272_9_std,272_9_delta,272_9_slope,835_0_last,835_0_mean,835_0_std,835_0_delta,835_0_slope,370_0_last,370_0_mean,370_0_std,370_0_delta,370_0_slope,291_0_last,291_0_mean,291_0_std,291_0_delta,291_0_slope,291_1_last,291_1_mean,291_1_std,291_1_delta,291_1_slope,291_2_last,291_2_mean,291_2_std,291_2_delta,291_2_slope,291_3_last,291_3_mean,291_3_std,291_3_delta,291_3_slope,291_4_last,291_4_mean,291_4_std,291_4_delta,291_4_slope,291_5_last,291_5_mean,291_5_std,291_5_delta,291_5_slope,291_6_last,291_6_mean,291_6_std,291_6_delta,291_6_slope,291_7_last,291_7_mean,291_7_std,291_7_delta,291_7_slope,291_8_last,291_8_mean,291_8_std,291_8_delta,291_8_slope,291_9_last,291_9_mean,291_9_std,291_9_delta,291_9_slope,291_10_last,291_10_mean,291_10_std,291_10_delta,291_10_slope,158_0_last,158_0_mean,158_0_std,158_0_delta,158_0_slope,158_1_last,158_1_mean,158_1_std,158_1_delta,158_1_slope,158_2_last,158_2_mean,158_2_std,158_2_delta,158_2_slope,158_3_last,158_3_mean,158_3_std,158_3_delta,158_3_slope,158_4_last,158_4_mean,158_4_std,158_4_delta,158_4_slope,158_5_last,158_5_mean,158_5_std,158_5_delta,158_5_slope,158_6_last,158_6_mean,158_6_std,158_6_delta,158_6_slope,158_7_last,158_7_mean,158_7_std,158_7_delta,158_7_slope,158_8_last,158_8_mean,158_8_std,158_8_delta,158_8_slope,158_9_last,158_9_mean,158_9_std,158_9_delta,158_9_slope,100_0_last,100_0_mean,100_0_std,100_0_delta,100_0_slope,459_0_last,459_0_mean,459_0_std,459_0_delta,459_0_slope,459_1_last,459_1_mean,459_1_std,459_1_delta,459_1_slope,459_2_last,459_2_mean,459_2_std,459_2_delta,459_2_slope,459_3_last,459_3_mean,459_3_std,459_3_delta,459_3_slope,459_4_last,459_4_mean,459_4_std,459_4_delta,459_4_slope,459_5_last,459_5_mean,459_5_std,459_5_delta,459_5_slope,459_6_last,459_6_mean,459_6_std,459_6_delta,459_6_slope,459_7_last,459_7_mean,459_7_std,459_7_delta,459_7_slope,459_8_last,459_8_mean,459_8_std,459_8_delta,459_8_slope,459_9_last,459_9_mean,459_9_std,459_9_delta,459_9_slope,459_10_last,4

In [14]:
# Temporal aggregates and labels for test set
sensorcols = df_testop.iloc[:, 2:].columns
sensors = [col for col in sensorcols]
df_testop_temporal = add_temporal_aggregates(testop_idx, sensors)
df_testop_temporal = df_testop_temporal.join(df_testspec)
mask = df_testop_temporal.groupby(level=0).cumcount(ascending=False) == 0
df_testop_temporal['class_label'] = np.nan
df_testop_temporal.loc[mask, 'class_label'] = np.asarray(df_testlab['class_label'])

for col in df_testspec.columns:
    df_testop_temporal[col] = df_testop_temporal[col].astype('category')

print(df_testop_temporal.memory_usage(deep=True).sum() / 1e9, "GB")
df_testop_temporal.head()

0.754525567 GB


,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,167_4,167_5,167_6,167_7,167_8,167_9,309_0,272_0,272_1,272_2,272_3,272_4,272_5,272_6,272_7,272_8,272_9,835_0,370_0,291_0,291_1,291_2,291_3,291_4,291_5,291_6,291_7,291_8,291_9,291_10,158_0,158_1,158_2,158_3,158_4,158_5,158_6,158_7,158_8,158_9,100_0,459_0,459_1,459_2,459_3,459_4,459_5,459_6,459_7,459_8,459_9,459_10,459_11,459_12,459_13,459_14,459_15,459_16,459_17,459_18,459_19,397_0,397_1,397_2,397_3,397_4,397_5,397_6,397_7,397_8,397_9,397_10,397_11,397_12,397_13,397_14,397_15,397_16,397_17,397_18,397_19,397_20,397_21,397_22,397_23,397_24,397_25,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35,171_0_last,171_0_mean,171_0_std,171_0_delta,171_0_slope,666_0_last,666_0_mean,666_0_std,666_0_delta,666_0_slope,427_0_last,427_0_mean,427_0_std,427_0_delta,427_0_slope,837_0_last,837_0_mean,837_0_std,837_0_delta,837_0_slope,167_0_last,167_0_mean,167_0_std,167_0_delta,167_0_slope,167_1_last,167_1_mean,167_1_std,167_1_delta,167_1_slope,167_2_last,167_2_mean,167_2_std,167_2_delta,167_2_slope,167_3_last,167_3_mean,167_3_std,167_3_delta,167_3_slope,167_4_last,167_4_mean,167_4_std,167_4_delta,167_4_slope,167_5_last,167_5_mean,167_5_std,167_5_delta,167_5_slope,167_6_last,167_6_mean,167_6_std,167_6_delta,167_6_slope,167_7_last,167_7_mean,167_7_std,167_7_delta,167_7_slope,167_8_last,167_8_mean,167_8_std,167_8_delta,167_8_slope,167_9_last,167_9_mean,167_9_std,167_9_delta,167_9_slope,309_0_last,309_0_mean,309_0_std,309_0_delta,309_0_slope,272_0_last,272_0_mean,272_0_std,272_0_delta,272_0_slope,272_1_last,272_1_mean,272_1_std,272_1_delta,272_1_slope,272_2_last,272_2_mean,272_2_std,272_2_delta,272_2_slope,272_3_last,272_3_mean,272_3_std,272_3_delta,272_3_slope,272_4_last,272_4_mean,272_4_std,272_4_delta,272_4_slope,272_5_last,272_5_mean,272_5_std,272_5_delta,272_5_slope,272_6_last,272_6_mean,272_6_std,272_6_delta,272_6_slope,272_7_last,272_7_mean,272_7_std,272_7_delta,272_7_slope,272_8_last,272_8_mean,272_8_std,272_8_delta,272_8_slope,272_9_last,272_9_mean,272_9_std,272_9_delta,272_9_slope,835_0_last,835_0_mean,835_0_std,835_0_delta,835_0_slope,370_0_last,370_0_mean,370_0_std,370_0_delta,370_0_slope,291_0_last,291_0_mean,291_0_std,291_0_delta,291_0_slope,291_1_last,291_1_mean,291_1_std,291_1_delta,291_1_slope,291_2_last,291_2_mean,291_2_std,291_2_delta,291_2_slope,291_3_last,291_3_mean,291_3_std,291_3_delta,291_3_slope,291_4_last,291_4_mean,291_4_std,291_4_delta,291_4_slope,291_5_last,291_5_mean,291_5_std,291_5_delta,291_5_slope,291_6_last,291_6_mean,291_6_std,291_6_delta,291_6_slope,291_7_last,291_7_mean,291_7_std,291_7_delta,291_7_slope,291_8_last,291_8_mean,291_8_std,291_8_delta,291_8_slope,291_9_last,291_9_mean,291_9_std,291_9_delta,291_9_slope,291_10_last,291_10_mean,291_10_std,291_10_delta,291_10_slope,158_0_last,158_0_mean,158_0_std,158_0_delta,158_0_slope,158_1_last,158_1_mean,158_1_std,158_1_delta,158_1_slope,158_2_last,158_2_mean,158_2_std,158_2_delta,158_2_slope,158_3_last,158_3_mean,158_3_std,158_3_delta,158_3_slope,158_4_last,158_4_mean,158_4_std,158_4_delta,158_4_slope,158_5_last,158_5_mean,158_5_std,158_5_delta,158_5_slope,158_6_last,158_6_mean,158_6_std,158_6_delta,158_6_slope,158_7_last,158_7_mean,158_7_std,158_7_delta,158_7_slope,158_8_last,158_8_mean,158_8_std,158_8_delta,158_8_slope,158_9_last,158_9_mean,158_9_std,158_9_delta,158_9_slope,100_0_last,100_0_mean,100_0_std,100_0_delta,100_0_slope,459_0_last,459_0_mean,459_0_std,459_0_delta,459_0_slope,459_1_last,459_1_mean,459_1_std,459_1_delta,459_1_slope,459_2_last,459_2_mean,459_2_std,459_2_delta,459_2_slope,459_3_last,459_3_mean,459_3_std,459_3_delta,459_3_slope,459_4_last,459_4_mean,459_4_std,459_4_delta,459_4_slope,459_5_last,459_5_mean,459_5_std,459_5_delta,459_5_slope,459_6_last,459_6_mean,459_6_std,459_6_delta,459_6_slope,459_7_last,459_7_mean,459_7_std,459_7_delta,459_7_slope,459_8_last,459_8_mean,459_8_std,459_8_delta,459_8_slope,459_9_last,459_9_mean,459_9_std,459_9_delta,459_9_slope,459_10_last,4

In [ ]:
# Save DataFrames as new feature-engineered CSV files

# Temporal aggregate train readouts
df_trainop_temporal.reset_index().to_parquet(
    "../data/fe_data/temporal_train.parquet",
    index=False
)

# Temporal aggregate val set
df_valop_temporal.reset_index().to_parquet(
    "../data/fe_data/temporal_val.parquet",
    index=False
)

# Temporal aggregate test set
df_testop_temporal.reset_index().to_parquet(
    "../data/fe_data/temporal_test.parquet",
    index=False
)